<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/ESAA_OB_WEEK02_1_review2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**주제**

감귤나무의 나무 생육 상태, 엽록소 및 새순 정보로부터 감귤 착과량을 회귀 예측  
https://dacon.io/competitions/official/236038/overview/description

##**데이터**
1. train.csv
- ID : 과수나무 고유 ID
- 착과량(int) : 실제 감귤 착과량 (Target)
- 나무 생육 상태 Features (5개)  
  - 수고(m), 수관폭1(min), 수관폭2(max), 수관폭평균(수관폭1과 수관폭2의 평균)
  - 데이터 기입은 cm 단위
- 새순 Features (89개)
  - 2022년 09월 01일 ~ 2022년 11월 28일에 일별 측정된 새순 데이터
- 엽록소 Features (89개)
  - 2022년 09월 01일 ~ 2022년 11월 28일에 일별 측정된 엽록소 데이터

2. test.csv
- ID : 과수나무 고유 ID
- 나무 생육 상태 Features (5개)
  - 수고(m), 수관폭1(min), 수관폭2(max), 수관폭평균(수관폭1과 수관폭2의 평균)
  - 데이터 기입은 cm 단위
- 새순 Features (89개)
  - 2022년 09월 01일 ~ 2022년 11월 28일에 일별 측정된 새순 데이터
- 엽록소 Features (89개)
  - 2022년 09월 01일 ~ 2022년 11월 28일에 일별 측정된 엽록소 데이터

3. sample_submission.csv
- ID : 과수나무 고유 ID
- 착과량(int) : 예측한 감귤 착과량

##**코드 리뷰**
[EDA]  
- 새순 시계열과 착과량의 상관 관계 분석
  - 새순 수와 착과량 반비례
- 엽록소(새순대비) 시계열과 착과량의 상관 관계 분석
  - 엽록소 수 많을수록 좋은 착과량
- 나무 형상은 대체적으로 착과량 상관관계 존재 X

[전처리]
- 새순 시계열 데이터
  - 새순 시계열 - 새순 초기값
  - 새순 시계열 - 새순 중기값
- 엽록소 시계열 데이터 / (새순 시계열 + 0.1)
- 나무의 형상 제외

[Training & Inference]
- DL의 경우 문제를 풀기 위한 정형화 구조가 없어 구조 단계부터 설계가 필요함
- 정형화된 구조가 있는 Tree-based Model 선택
- Hyperparameters Tuning Tools을 사용하기 전 trivial tuning 시도
  - Random Forest, LGBM, XGBoost 성능차이가 크지 않음
  - 가장 간단한 Random Forest 선택
- Hyperparmaeters Tuning for RE
  - Split into Training and Validation Data(6:4)
  - Hyperparameters for RF
    - tree의 수는 200개 정도면 충분
    - feature sampling 수는 sqrt(m)이면 충분
  - Decision
    - max depth: 5일 때 validation NMAE 최소

##**배울점**
- EDA의 결과를 보고 실제 전처리에 잘 반영하였다.
  - 새순 같은 경우, 수와 착과량이 반비례하므로 새순 원값만 이용한 것이 아니라 초기값/중기값 대비 변화량을 생성하였다.
  - 엽록소 단독이 아닌 새순 대비 엽록소 비율을 생성하였다.
- DL과 ML을 비교하고 ML에서도 모델끼리 먼저 trivial tuning을 통해 성능차이를 미리 확인했다. 이로 인해 모델을 선택하는 데 드는 시간을 줄이고 효율성을 높였다.
- 시계열 데이터를 모델이 쓰기 쉬운 형태로 변환했다.
  - 원본: wide
  - melt()로 long 형태 변환 -> 날짜와 변수타입 분리 -> pivot()으로 wide 형태 복원

In [ ]:
# EDA 반영
train_melt_preproc['새순diff0'] = train_melt_preproc['새순']-train_melt_preproc['새순0']
train_melt_preproc['새순diff1'] = train_melt_preproc['새순']-train_melt_preproc['새순1']
train_melt_preproc['엽록소_새순'] = train_melt_preproc['엽록소']/(train_melt_preproc['새순']+0.1)

In [ ]:
# 시계열 데이터 변환
train_melt = train_raw_df.melt(id_vars=['ID']+y_var+id_var, value_vars=saesoon+yuprokso)
train_melt['date'] = train_melt['variable'].apply(lambda x: x.split(' ')[0])
train_melt['type'] = train_melt['variable'].apply(lambda x: x.split(' ')[1])

In [ ]:
# 다시 wide 형태로 복원
train_df = train_melt_preproc.pivot(
    index=['ID']+y_var+id_var,
    columns=['date'],
    values=['새순diff0','새순diff1','새순', '엽록소_새순']
).reset_index()